<a href="https://colab.research.google.com/github/singh-himanshu3/CV_Project/blob/main/CV_AI_TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup, Weights, and Clean Dataset Generation (Foolproof Version)
import os
import torch

print("1. Cloning SCUNet Repository...")
!git clone https://github.com/cszn/SCUNet.git
%cd SCUNet

print("\n2. Installing required libraries...")
!pip install timm einops thop scikit-image

print("\n3. Downloading pre-trained backbone weights...")
os.makedirs('model_zoo', exist_ok=True)
!wget -q -nc https://github.com/cszn/KAIR/releases/download/v1.0/scunet_color_real_psnr.pth -P model_zoo/

print("\n4. Generating 'Clean' Training Dataset locally (Bypassing internet blocks)...")

import cv2
from skimage import data
os.makedirs('clean_dataset', exist_ok=True)


clean_images = {
    'astronaut.png': data.astronaut(),
    'cat.png': data.chelsea(),
    'coffee.png': data.coffee(),
    'retina.png': data.retina(),
    'cells.png': data.immunohistochemistry()
}

for name, img in clean_images.items():

    cv2.imwrite(f'clean_dataset/{name}', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

print(f"Generated {len(clean_images)} high-quality clean images for training!")


if torch.cuda.is_available():
    print(f"\n Success! GPU active: {torch.cuda.get_device_name(0)}")
    print("SCUNet Repo cloned, libraries installed, and Clean Dataset is ready!")
else:
    print("\nERROR: No GPU found. Go to Runtime > Change runtime type and select T4 GPU.")

In [ ]:
# Cell 2: Connecting to Google Drive for Large Datasets
from google.colab import drive
import os

print(" Requesting access to Google Drive...")

drive.mount('/content/drive')

print("\n Google Drive successfully mounted!")

dataset_path = '/content/drive/MyDrive/div2k.zip'

if os.path.exists(dataset_path):
    print(f"Found your dataset at: {dataset_path}")
    print("Extracting it to the fast local Colab disk for training...")

    !unzip -q -o {dataset_path} -d /content/clean_dataset_custom/
    print("Extraction complete! We are ready to synthesize noise.")
else:
    print(f" Could not find {dataset_path}.")
    print("Make sure the spelling matches exactly what is in your Google Drive!")

In [ ]:
# Cell 3: The Dynamic CCTV/Weather Data Synthesizer
import os
import glob
import cv2
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

class CCTVForensicDataset(Dataset):
    def __init__(self, root_dir, patch_size=128):

        print("Searching for images...")
        self.image_paths = glob.glob(os.path.join(root_dir, '**', '*.png'), recursive=True)
        self.patch_size = patch_size
        print(f"Found {len(self.image_paths)} clean images for training!")

    def __len__(self):
        return len(self.image_paths)

    def synthesize_cctv_noise(self, clean_patch):
        noisy_patch = clean_patch.copy().astype(np.float32)


        noise_level = random.randint(15, 55)
        noise = np.random.normal(0, noise_level, noisy_patch.shape)
        noisy_patch = noisy_patch + noise

        if random.random() < 0.5:

            num_streaks = random.randint(10, 40)
            for _ in range(num_streaks):
                x1 = random.randint(0, self.patch_size)
                y1 = random.randint(0, self.patch_size)
                x2 = x1 + random.randint(-10, 10)
                y2 = y1 + random.randint(10, 30)
                color = random.randint(150, 255)
                thickness = random.randint(1, 2)
                cv2.line(noisy_patch, (x1, y1), (x2, y2), (color, color, color), thickness)


        noisy_patch = np.clip(noisy_patch, 0, 255).astype(np.uint8)
        return noisy_patch

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]
        clean_img = cv2.imread(img_path)
        clean_img = cv2.cvtColor(clean_img, cv2.COLOR_BGR2RGB)


        h, w, _ = clean_img.shape
        y = random.randint(0, h - self.patch_size - 1)
        x = random.randint(0, w - self.patch_size - 1)
        clean_patch = clean_img[y:y+self.patch_size, x:x+self.patch_size]


        if random.random() < 0.5:
            clean_patch = cv2.flip(clean_patch, random.choice([-1, 0, 1]))


        noisy_patch = self.synthesize_cctv_noise(clean_patch)


        clean_tensor = torch.from_numpy(np.transpose(clean_patch, (2, 0, 1))).float() / 255.0
        noisy_tensor = torch.from_numpy(np.transpose(noisy_patch, (2, 0, 1))).float() / 255.0

        return noisy_tensor, clean_tensor


dataset_dir = '/content/clean_dataset_custom/'
train_dataset = CCTVForensicDataset(root_dir=dataset_dir, patch_size=128)


train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)


noisy_batch, clean_batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
plt.suptitle("Synthesized Training Pairs: Noisy CCTV (Top) vs. Clean Target (Bottom)", fontsize=16)

for i in range(4):

    ax_noisy = axes[0, i]
    ax_noisy.imshow(np.transpose(noisy_batch[i].numpy(), (1, 2, 0)))
    ax_noisy.axis('off')


    ax_clean = axes[1, i]
    ax_clean.imshow(np.transpose(clean_batch[i].numpy(), (1, 2, 0)))
    ax_clean.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: The Transfer Learning Training Loop
import torch
import torch.nn as nn
import torch.optim as optim
from models.network_scunet import SCUNet
from tqdm.notebook import tqdm
import os


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading SCUNet onto {device}...")
model = SCUNet(in_nc=3, config=[4,4,4,4,4,4,4], dim=64)
model.load_state_dict(torch.load('model_zoo/scunet_color_real_psnr.pth'), strict=True)

for param in model.parameters():
    param.requires_grad = False

for param in model.m_up3.parameters():
    param.requires_grad = True
for param in model.m_tail.parameters():
    param.requires_grad = True

model = model.to(device)


criterion = nn.L1Loss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)


epochs = 1

max_batches = 100

print("\n STARTING TRANSFER LEARNING ")
print("Teaching the AI to remove CCTV Noise and Synthetic Rain...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0


    progress_bar = tqdm(enumerate(train_loader), total=min(len(train_loader), max_batches), desc=f"Training Progress")

    for i, (noisy_tensor, clean_tensor) in progress_bar:
        if i >= max_batches:
            break

        noisy_tensor = noisy_tensor.to(device)
        clean_tensor = clean_tensor.to(device)


        optimizer.zero_grad()


        outputs = model(noisy_tensor)


        loss = criterion(outputs, clean_tensor)


        loss.backward()
        optimizer.step()


        progress_bar.set_postfix({'L1 Loss': f"{loss.item():.4f}"})

save_path = '/content/drive/MyDrive/scunet_cctv_weather_finetuned.pth'
torch.save(model.state_dict(), save_path)

print(f"\n TRAINING COMPLETE! ")
print(f"Your custom fine-tuned weights are permanently saved to your Google Drive at: {save_path}")
print("You can now load this .pth file into your Inference/Video project instead of the default weights!")